In [ ]:
import numpy as np
import pandas as pd
import pulp
import folium
from math import radians, sin, cos, sqrt, atan2
import matplotlib.pyplot as plt
from IPython.display import display

In [2]:

# 1. KHỞI TẠO DỮ LIỆU MẪU (TỈ LỆ THU NHỎ)
np.random.seed(42)

HCMC_LAT, HCMC_LON = 10.7769, 106.7009
NUM_CLIENTS = 12
NUM_DEPOTS = 3
VEHICLE_CAPACITY = 35  # Sức chở của 1 xe tải

clients = [f"C{i}" for i in range(1, NUM_CLIENTS + 1)]
depots = [f"D{j}" for j in range(1, NUM_DEPOTS + 1)]

# Tọa độ và Nhu cầu khách hàng
client_data = {
    i: {
        "lat": HCMC_LAT + np.random.uniform(-0.02, 0.02),
        "lon": HCMC_LON + np.random.uniform(-0.02, 0.02),
        "demand": np.random.randint(4, 12)
    } for i in clients
}

# Tọa độ và Chi phí kho
depot_data = {
    j: {
        "lat": HCMC_LAT + np.random.uniform(-0.02, 0.02),
        "lon": HCMC_LON + np.random.uniform(-0.02, 0.02),
        "capacity": np.random.randint(80, 120),
        "fixed_cost": np.random.randint(200, 400)
    } for j in depots
}

# Hàm tính khoảng cách
def dist(loc1, loc2):
    lat1, lon1 = map(radians, [loc1["lat"], loc1["lon"]])
    lat2, lon2 = map(radians, [loc2["lat"], loc2["lon"]])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 6371.0 * 2 * atan2(sqrt(a), sqrt(1-a))

# Lấy tọa độ gộp (All Nodes)
nodes = {**client_data, **depot_data}
c = { (u, v): dist(nodes[u], nodes[v]) for u in nodes for v in nodes }



In [ ]:
# 2. XÂY DỰNG MÔ HÌNH LRP (Location-Routing Problem)
model = pulp.LpProblem("Location_Routing_Problem", pulp.LpMinimize)

# Biến y[k]: Có mở kho k không?
y = pulp.LpVariable.dicts("Open", depots, cat="Binary")

# Biến z[i][k]: Khách hàng i có được gán cho kho k không?
z = pulp.LpVariable.dicts("Assign", (clients, depots), cat="Binary")

# Biến x[u][v][k]: Xe xuất phát từ kho k đi trực tiếp từ u sang v
# u, v có thể là khách hàng hoặc chính kho k đó
x = pulp.LpVariable.dicts("Route", 
                          [(u, v, k) for k in depots 
                           for u in clients + [k] 
                           for v in clients + [k]], 
                          cat="Binary")

# Biến u[i]: Tải trọng tích lũy của xe khi rời điểm i 
u_var = pulp.LpVariable.dicts("Load", clients, lowBound=0, upBound=VEHICLE_CAPACITY, cat="Continuous")

In [3]:
# 3. HÀM MỤC TIÊU & RÀNG BUỘC
# Mục tiêu: Tối thiểu hóa Chi phí mở kho + Chi phí quãng đường xe chạy (Trọng số 1.5/km)
model += (
    pulp.lpSum(depot_data[k]["fixed_cost"] * y[k] for k in depots) +
    1.5 * pulp.lpSum(c[u,v] * x[u,v,k] 
                     for k in depots for u in clients + [k] for v in clients + [k] if u != v)
)

# Ràng buộc 1: Mỗi khách hàng được gán cho đúng 1 kho
for i in clients:
    model += pulp.lpSum(z[i][k] for k in depots) == 1

# Ràng buộc 2: Khách hàng chỉ được gán cho kho k nếu kho k được mở
for i in clients:
    for k in depots:
        model += z[i][k] <= y[k]

# Ràng buộc 3: Tổng nhu cầu gán vào kho k không vượt quá sức chứa
for k in depots:
    model += pulp.lpSum(client_data[i]["demand"] * z[i][k] for i in clients) <= depot_data[k]["capacity"] * y[k]

# Ràng buộc 4: Flow Conservation (Định tuyến)
# Xe đi vào điểm i (từ mạng lưới của kho k) phải bằng xe đi ra khỏi i
for k in depots:
    for i in clients:
        model += pulp.lpSum(x[u, i, k] for u in clients + [k] if u != i) == z[i][k]
        model += pulp.lpSum(x[i, v, k] for v in clients + [k] if v != i) == z[i][k]

# Ràng buộc 5: MTZ Subtour Elimination & Vehicle Capacity
# Ngăn xe đi vòng quanh các khách hàng mà không chịu quay về kho, đồng thời giới hạn sức chở của xe
for k in depots:
    for i in clients:
        for j in clients:
            if i != j:
                model += u_var[i] - u_var[j] + VEHICLE_CAPACITY * x[i, j, k] <= VEHICLE_CAPACITY - client_data[j]["demand"]

# Tải trọng tối thiểu khi tới một điểm ít nhất phải bằng nhu cầu điểm đó
for i in clients:
    model += u_var[i] >= client_data[i]["demand"]

# Chạy Solver
print("Đang giải hệ phương trình LRP (Vui lòng đợi vài giây)...")
model.solve(pulp.PULP_CBC_CMD(msg=False))
print(f"Trạng thái: {pulp.LpStatus[model.status]}")
print(f"Tổng chi phí tối ưu: {pulp.value(model.objective):.2f}")


C:\Users\Huy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pulp\pulp.py:2178: UserWarning: Overwriting previously set objective.
  warnings.warn("Overwriting previously set objective.")


Đang giải hệ phương trình LRP (Vui lòng đợi vài giây)...
Trạng thái: Optimal
Tổng chi phí tối ưu: 249.82


In [ ]:
# Vẽ Khách hàng và gắn nhãn tên trực tiếp lên bản đồ
from folium import plugins
# VẼ LỘ TRÌNH CÓ HƯỚNG DI CHUYỂN (ANIMATED ROUTING)
for u, v, k in active_routes:
    loc_u = [nodes[u]["lat"], nodes[u]["lon"]]
    loc_v = [nodes[v]["lat"], nodes[v]["lon"]]
    
    # Sử dụng AntPath
    plugins.AntPath(
        locations=[loc_u, loc_v],
        color=colors[k],        # Màu của lộ trình theo từng kho
        weight=3.5,
        opacity=0.8,
        dash_array=[10, 15],    # Khoảng cách giữa các đốm sáng
        delay=800,              # Tốc độ di chuyển (càng nhỏ càng nhanh)
        pulse_color='white',    # Màu của luồng sáng di chuyển
        tooltip=f"Từ {u} ➔ {v}"
    ).add_to(m)
for i in clients:
    assigned_k = next((k for k in depots if pulp.value(z[i][k]) > 0.5), None)
    c_color = colors[assigned_k] if assigned_k else "gray"
    
    # 1. Vẽ Dấu chấm tròn (Dot)
    folium.CircleMarker(
        [client_data[i]["lat"], client_data[i]["lon"]],
        radius=6, 
        color=c_color, 
        fill=True, 
        fill_opacity=0.9,
        popup=f"Khách: {i}<br>Nhu cầu: {client_data[i]['demand']}"
    ).add_to(m)
    
    # 2. Thêm Nhãn tên (Label) hiển thị cố định sát bên cạnh dấu chấm
    folium.Marker(
        [client_data[i]["lat"], client_data[i]["lon"]],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-family: Arial, sans-serif;
                font-size: 12px;
                font-weight: bold;
                color: {c_color};
                text-shadow: 1.5px 1.5px 2px white, -1.5px -1.5px 2px white, 1.5px -1.5px 2px white, -1.5px 1.5px 2px white;
                transform: translate(10px, -12px);
                white-space: nowrap;
            ">
                {i}
            </div>
            """
        )
    ).add_to(m)
display(m)

# BỔ SUNG: TRÍCH XUẤT CHÍNH XÁC THỨ TỰ LỘ TRÌNH (SEQUENCE)
print("\n" + "="*50)
print("CHI TIẾT THỨ TỰ LỘ TRÌNH XE CHẠY (DISPATCH PLAN)")
print("="*50)

for k in depots:
    # Chỉ xét các kho được mở
    if pulp.value(y[k]) > 0.5: 
        # Tìm tất cả các điểm xuất phát trực tiếp từ kho k (Khởi đầu chuyến đi)
        start_edges = [(u, v) for (u, v, depot) in active_routes if depot == k and u == k]
        
        route_count = 1
        for start_edge in start_edges:
            current_node = start_edge[0] # Chính là kho k
            next_node = start_edge[1]    # Khách hàng đầu tiên
            
            route_seq = [current_node, next_node]
            route_load = client_data[next_node]["demand"] if next_node in clients else 0
            
            # Liên tục dò mũi tên tiếp theo cho đến khi đầu kim chỉ về lại kho k
            while next_node != k:
                # Tìm cạnh bắt đầu từ next_node
                next_edge = next((u, v) for (u, v, depot) in active_routes if depot == k and u == next_node)
                next_node = next_edge[1]
                route_seq.append(next_node)
                
                # Cộng dồn tải trọng nếu điểm đến là khách hàng
                if next_node in clients:
                    route_load += client_data[next_node]["demand"]
            
            # In ra lộ trình hoàn chỉnh của chuyến xe đó
            sequence_str = " ➔ ".join(route_seq)
            print(f"📍 Kho {k} | Xe {route_count} | Tải trọng: {route_load}/{VEHICLE_CAPACITY}")
            print(f"   Lộ trình: {sequence_str}\n")
            
            route_count += 1


CHI TIẾT THỨ TỰ LỘ TRÌNH XE CHẠY (DISPATCH PLAN)
📍 Kho D1 | Xe 1 | Tải trọng: 32/35
   Lộ trình: D1 ➔ C4 ➔ C3 ➔ C8 ➔ C7 ➔ D1

📍 Kho D1 | Xe 2 | Tải trọng: 35/35
   Lộ trình: D1 ➔ C5 ➔ C1 ➔ C2 ➔ C12 ➔ C10 ➔ D1

📍 Kho D1 | Xe 3 | Tải trọng: 24/35
   Lộ trình: D1 ➔ C11 ➔ C6 ➔ C9 ➔ D1

